In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType

In [0]:
spark.sql("USE CATALOG mvp_saude")
spark.sql("USE SCHEMA silver")

DataFrame[]

In [0]:
print("🔄 Lendo dados brutos da camada BRONZE...")
df_bronze = spark.table("mvp_saude.bronze.sih_raw")

🔄 Lendo dados brutos da camada BRONZE...


In [0]:
for col_name in df_bronze.columns:
    df_bronze = df_bronze.withColumnRenamed(col_name, col_name.upper())

In [0]:
df_silver = df_bronze

# Garantindo o filtro RS

In [0]:
df_silver = df_silver.withColumn("uf_sigla", F.lit("RS"))

# TIPAGEM E PADRONIZAÇÃO DE COLUNAS CHAVE

# Valor Total da Internação (VAL_TOT)

In [0]:
if "VAL_TOT" in df_silver.columns:
    df_silver = df_silver.withColumn("val_total_internacao", F.col("VAL_TOT").cast(DoubleType()))
elif "VALOR_TOTAL" in df_silver.columns:
    df_silver = df_silver.withColumn("val_total_internacao", F.col("VALOR_TOTAL").cast(DoubleType()))
else:
    df_silver = df_silver.withColumn("val_total_internacao", F.lit(0.0))

# Dias de Permanência (DIAS_PERM)

In [0]:
if "DIAS_PERM" in df_silver.columns:
    df_silver = df_silver.withColumn("dias_permanencia", F.col("DIAS_PERM").cast(IntegerType()))
else:
    df_silver = df_silver.withColumn("dias_permanencia", F.lit(1))

# Idade do Paciente

In [0]:
if "IDADE" in df_silver.columns:
    df_silver = df_silver.withColumn("idade_paciente", F.col("IDADE").cast(IntegerType()))
else:
    df_silver = df_silver.withColumn("idade_paciente", F.lit(0))

# Indicador de Óbito (MORTE / OBITO)

In [0]:
if "MORTE" in df_silver.columns:
    df_silver = df_silver.withColumn("ind_obito", F.col("MORTE").cast(IntegerType()))
elif "OBITO" in df_silver.columns:
    df_silver = df_silver.withColumn("ind_obito", F.col("OBITO").cast(IntegerType()))
else:
    df_silver = df_silver.withColumn("ind_obito", F.lit(0))

# Diagnóstico Principal (CID / DIAG_PRINC)

In [0]:
if "DIAG_PRINC" in df_silver.columns:
    df_silver = df_silver.withColumn("cod_cid_principal", F.col("DIAG_PRINC").cast(StringType()))
elif "CID" in df_silver.columns:
    df_silver = df_silver.withColumn("cod_cid_principal", F.col("CID").cast(StringType()))
else:
    df_silver = df_silver.withColumn("cod_cid_principal", F.lit("IGNORADO"))

# Município de Residência

In [0]:
if "UF_ZI" in df_silver.columns:
    df_silver = df_silver.withColumn("cod_municipio", F.col("UF_ZI").cast(StringType()))
elif "MUNIC_RES" in df_silver.columns:
    df_silver = df_silver.withColumn("cod_municipio", F.col("MUNIC_RES").cast(StringType()))
else:
    df_silver = df_silver.withColumn("cod_municipio", F.lit("430000"))

# REGRAS DE QUALIDADE DE DADOS (DATA QUALITY)

In [0]:
df_silver_clean = df_silver \
    .filter(F.col("val_total_internacao") >= 0) \
    .filter(F.col("dias_permanencia") >= 0) \
    .fillna({"ind_obito": 0, "cod_cid_principal": "N/A"}) \
    .withColumn("uf_sigla", F.lit("RS")) \
    .withColumn("dh_processamento_silver", F.current_timestamp())

# Grava a tabela limpa em formato Delta Lake

In [0]:
df_silver_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_saude.silver.sih_cleaned")

# Validação

In [0]:
%sql
SELECT 
  uf_sigla,
  cod_municipio,
  cod_cid_principal,
  idade_paciente,
  dias_permanencia,
  val_total_internacao,
  ind_obito,
  dh_processamento_silver
FROM mvp_saude.silver.sih_cleaned
LIMIT 10;

uf_sigla,cod_municipio,cod_cid_principal,idade_paciente,dias_permanencia,val_total_internacao,ind_obito,dh_processamento_silver
RS,431720,K409,59,1,547.51,0,2026-09-25T23:52:54.518Z
RS,431720,K409,61,1,551.51,0,2026-09-25T23:52:54.518Z
RS,431720,M235,23,1,2574.76,0,2026-09-25T23:52:54.518Z
RS,431720,C180,73,2,1108.0,0,2026-09-25T23:52:54.518Z
RS,431720,J180,49,5,3759.47,0,2026-09-25T23:52:54.518Z
RS,431720,K564,85,2,491.62,1,2026-09-25T23:52:54.518Z
RS,431720,I64,77,4,3557.67,0,2026-09-25T23:52:54.518Z
RS,431720,S960,46,1,680.2,0,2026-09-25T23:52:54.518Z
RS,431720,M702,65,1,258.41,0,2026-09-25T23:52:54.518Z
RS,431720,O808,29,1,516.6,0,2026-09-25T23:52:54.518Z


# Trazendo tabelas de Cidades e CID's

In [0]:
import pandas as pd
import requests
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
spark.sql("USE CATALOG mvp_saude")
spark.sql("USE SCHEMA silver")

DataFrame[]

In [0]:
url_ibge_rs = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios"
response = requests.get(url_ibge_rs)
data_muni = response.json()

muni_list = [
    ("430000", "4300000", "RS - Município Não Informado / Ignorado", "RS"),
    ("43000", "4300000", "RS - Município Não Informado / Ignorado", "RS")
]

for item in data_muni:
    cod_full = str(item.get("id", "")).strip()
    cod_6 = cod_full[:6]
    nome = str(item.get("nome", "")).strip()
    muni_list.append((cod_6, cod_full, nome, "RS"))

schema_muni = StructType([
    StructField("cod_municipio", StringType(), True),
    StructField("cod_municipio_full", StringType(), True),
    StructField("nome_municipio", StringType(), True),
    StructField("uf_sigla", StringType(), True)
])

df_dim_muni = spark.createDataFrame(muni_list, schema=schema_muni).drop_duplicates(subset=['cod_municipio'])
df_dim_muni.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_saude.silver.dim_municipios")

In [0]:
df_cid_raw = spark.table("mvp_saude.default.cid_10_subcategorias")

In [0]:
col_unica = df_cid_raw.columns[0]

In [0]:
df_dim_cid_tratada = (
    df_cid_raw.withColumn("cols_split", F.split(F.col(f"`{col_unica}`"), ";"))
    .select(
        # Posição 0: Código do CID (SUBCAT)
        F.upper(
            F.trim(
                F.regexp_replace(
                    F.col("cols_split").getItem(0), "\\.", ""
                )
            )
        ).alias("cod_cid_principal"),
        # Posição 4: Descrição do CID
        F.trim(F.col("cols_split").getItem(4)).alias("descricao_cid"),
    )
    .filter(
        (F.col("cod_cid_principal") != "SUBCAT")
        & F.col("cod_cid_principal").isNotNull()
        & (F.col("cod_cid_principal") != "")
    )
    .withColumn(
        "descricao_cid",
        F.regexp_replace(F.col("descricao_cid"), "[?]+", ""),
    )
    .groupBy("cod_cid_principal")
    .agg(F.first("descricao_cid", ignorenulls=True).alias("descricao_cid"))
)

In [0]:
df_dim_cid_tratada.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("mvp_saude.silver.dim_cid10")

# Dimensão especialidades

In [0]:
dados_especialidades = [
    ("01", "Cirúrgica"),
    ("02", "Obstétrica"),
    ("03", "Clínica Médica"),
    ("04", "Pediatria"),
    ("05", "Psiquiatria"),
    ("06", "Tisiologia"),
    ("07", "Hanseníase"),
    ("08", "Pediatria Cirúrgica"),
    ("09", "Reabilitação"),
    ("10", "Psiquiatria em Hospital Geral")
]

df_espec = spark.createDataFrame(dados_especialidades, ["cod_especialidade", "descricao_especialidade"])
df_espec.write.format("delta").mode("overwrite").saveAsTable("mvp_saude.silver.dim_especialidades")

# Validação camada silver

In [0]:
import pyspark.sql.functions as F

spark.sql("USE CATALOG mvp_saude")
spark.sql("USE SCHEMA silver")

DataFrame[]

In [0]:
import pyspark.sql.functions as F

spark.sql("USE CATALOG mvp_saude")
spark.sql("USE SCHEMA silver")

DataFrame[]

In [0]:
df_sih = spark.table("mvp_saude.silver.sih_cleaned")
df_muni = spark.table("mvp_saude.silver.dim_municipios")
df_cid = spark.table("mvp_saude.silver.dim_cid10")

In [0]:
try:
    total_internacoes = df_sih.count()
    print(f"   • sih_cleaned (Fato)      : {total_internacoes:,} registros")
except Exception as e:
    print(f"   ⚠️ Erro ao contar sih_cleaned: {e}")
    total_internacoes = 0

try:
    total_municipios = df_muni.count()
    print(f"   • dim_municipios (Dim)    : {total_municipios:,} registros")
except Exception as e:
    print(f"   ⚠️ Erro ao contar dim_municipios: {e}")

try:
    total_cids = df_cid.count()
    print(f"   • dim_cid10 (Dim)         : {total_cids:,} diagnósticos")
except Exception as e:
    print(f"   ⚠️ Erro ao contar dim_cid10: {e}")

   • sih_cleaned (Fato)      : 63,922 registros
   • dim_municipios (Dim)    : 499 registros
   • dim_cid10 (Dim)         : 12,451 diagnósticos


In [0]:
stats = df_sih.select(
    F.count("*").alias("total"),
    F.round(F.sum("val_total_internacao"), 2).alias("custo_total"),
    F.round(F.avg("val_total_internacao"), 2).alias("custo_medio"),
    F.sum("ind_obito").alias("total_obitos")
).collect()[0]

In [0]:
df_sih.select("cod_municipio", "cod_cid_principal", "val_total_internacao", "ind_obito").show(5)

+-------------+-----------------+--------------------+---------+
|cod_municipio|cod_cid_principal|val_total_internacao|ind_obito|
+-------------+-----------------+--------------------+---------+
|       431720|             K409|              547.51|        0|
|       431720|             K409|              551.51|        0|
|       431720|             M235|             2574.76|        0|
|       431720|             C180|              1108.0|        0|
|       431720|             J180|             3759.47|        0|
+-------------+-----------------+--------------------+---------+
only showing top 5 rows
